In [ ]:
# Install BigQuery library
!pip install google-cloud-bigquery pandas --quiet

# Import libraries
from google.cloud import bigquery
from google.colab import auth
import pandas as pd

# Authenticate with your Google account
auth.authenticate_user()

# Connect to your BigQuery project
project_id = 'medicare-stars-2026'
client = bigquery.Client(project=project_id)

print("Connected to BigQuery successfully!")

Connected to BigQuery successfully!


In [ ]:
# Query 1 — Performance Categories
query1 = """
SELECT
  Contract_ID,
  Contract_Name,
  Organization_Type,
  Measure_Name,
  Star_Rating,
  Measure_Score,
  CASE
    WHEN SAFE_CAST(REPLACE(Star_Rating, '%', '') AS FLOAT64) < 60 THEN 'Below Threshold'
    WHEN SAFE_CAST(REPLACE(Star_Rating, '%', '') AS FLOAT64) BETWEEN 60 AND 75 THEN 'At Risk'
    WHEN SAFE_CAST(REPLACE(Star_Rating, '%', '') AS FLOAT64) > 75 THEN 'Performing Well'
    ELSE 'Unknown'
  END AS Performance_Category
FROM medicare-stars-2026.stars_data.measure_scores_clean
ORDER BY SAFE_CAST(REPLACE(Star_Rating, '%', '') AS FLOAT64) ASC
"""

df1 = client.query(query1).to_dataframe()
print(f"Total records: {len(df1)}")
print(df1.head(10))

Total records: 769
  Contract_ID                                      Contract_Name  \
0      H9977            DEVOTED HEALTH PLAN OF NEW MEXICO, INC.    
1      H9916         WELLCARE HEALTH INSURANCE OF ARIZONA, INC.    
2      H9802       DEVOTED HEALTH INSURANCE COMPANY OF NEBRASKA    
3      H9771         AETNA BETTER HEALTH PREMIER PLAN MMAI INC.    
4      H9623    Ventura County Medi-Cal Managed Care Commission    
5      H9326                              WYOBLUE ADVANTAGE INC    
6      H9314               AETNA BETTER HEALTH OF MICHIGAN INC.    
7      H8481          HAWAII MEDICAL SERVICE ASSOCIATION (HMSA)    
8      H8212           AMERIHEALTH CARITAS NORTH CAROLINA, INC.    
9      H8051   SAN FRANCISCO HEALTH AUTHORITY DBA SAN FRANCIS...   

  Organization_Type                                       Measure_Name  \
0        Local CCP                               Devoted Health, Inc.    
1        Local CCP                                Centene Corporation    
2        L

In [ ]:
# Query 2 — Measure Performance Analysis
query2 = """
SELECT
  Measure_Name,
  COUNT(Contract_ID) AS Total_Plans,
  ROUND(AVG(SAFE_CAST(REPLACE(Star_Rating, '%', '') AS FLOAT64)), 2) AS Avg_Score,
  ROUND(MIN(SAFE_CAST(REPLACE(Star_Rating, '%', '') AS FLOAT64)), 2) AS Min_Score,
  ROUND(MAX(SAFE_CAST(REPLACE(Star_Rating, '%', '') AS FLOAT64)), 2) AS Max_Score,
  COUNTIF(SAFE_CAST(REPLACE(Star_Rating, '%', '') AS FLOAT64) < 60) AS Plans_Below_Threshold
FROM medicare-stars-2026.stars_data.measure_scores_clean
GROUP BY Measure_Name
ORDER BY Avg_Score ASC
"""

df2 = client.query(query2).to_dataframe()
print(f"Total measures: {len(df2)}")
print(df2.head(10))

Total measures: 182
                                        Measure_Name  Total_Plans  Avg_Score  \
0   Ventura County Medi-Cal Managed Care Commission             1        NaN   
1                    SAN FRANCISCO HEALTH AUTHORITY             1        NaN   
2  SANTA BARBARA SAN LUIS OBISPO REGIONAL HEALTH ...            1        NaN   
3                        Elite Health Systems, Inc.             1        NaN   
4              SAN JOAQUIN COUNTY HEALTH COMMISSION             1        NaN   
5  SANTA CRUZ MONTEREY MERCED SAN BENITO MARIPOSA...            1        NaN   
6                      Contra Costa Health Services             1        NaN   
7                         Kern Health Systems (KHS)             1        NaN   
8                    DLP Marquette Health Plan, LLC             1        NaN   
9          NEIGHBORHOOD HEALTH PLAN OF RHODE ISLAND             2        NaN   

   Min_Score  Max_Score  Plans_Below_Threshold  
0        NaN        NaN                      0  
1

In [ ]:
# Check exact column values
query_check = """
SELECT
  string_field_0,
  string_field_1,
  string_field_2,
  string_field_3,
  string_field_4,
  string_field_5,
  string_field_6,
  string_field_7
FROM medicare-stars-2026.stars_data.measure_scores
WHERE string_field_0 NOT LIKE '2026%'
AND string_field_0 = 'H7323'
LIMIT 5
"""

df_check = client.query(query_check).to_dataframe()
print(df_check.to_string())

Empty DataFrame
Columns: [string_field_0, string_field_1, string_field_2, string_field_3, string_field_4, string_field_5, string_field_6, string_field_7]
Index: []


In [ ]:
# Check raw data structure
query_raw = """
SELECT
  string_field_0,
  string_field_1,
  string_field_2,
  string_field_3,
  string_field_4,
  string_field_5,
  string_field_6
FROM medicare-stars-2026.stars_data.measure_scores
LIMIT 10
"""

df_raw = client.query(query_raw).to_dataframe()
print(df_raw.to_string())

  string_field_0     string_field_1                                      string_field_2                      string_field_3           string_field_4                                        string_field_5                    string_field_6
0    CONTRACT_ID  Organization Type                                       Contract Name         Organization Marketing Name      Parent Organization  HD1: Staying Healthy: Screenings, Tests and Vaccines                              None
1           None               None                                                None                                None                     None                          C01: Breast Cancer Screening  C02: Colorectal Cancer Screening
2           None               None                                                None                                None                     None                               01/01/2024 – 12/31/2024           01/01/2024 – 12/31/2024
3         H7323          Local CCP          WELLCARE

In [ ]:
# Load full data directly into pandas — skip BigQuery view issues
query_full = """
SELECT *
FROM medicare-stars-2026.stars_data.measure_scores
WHERE string_field_0 NOT LIKE '2026%'
AND string_field_0 != 'CONTRACT_ID'
AND string_field_0 IS NOT NULL
AND string_field_1 IS NOT NULL
AND string_field_1 != 'Organization Type'
"""

df = client.query(query_full).to_dataframe()
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")
print(df.columns.tolist())

Total rows: 769
Total columns: 50
['string_field_0', 'string_field_1', 'string_field_2', 'string_field_3', 'string_field_4', 'string_field_5', 'string_field_6', 'string_field_7', 'string_field_8', 'string_field_9', 'string_field_10', 'string_field_11', 'string_field_12', 'string_field_13', 'string_field_14', 'string_field_15', 'string_field_16', 'string_field_17', 'string_field_18', 'string_field_19', 'string_field_20', 'string_field_21', 'string_field_22', 'string_field_23', 'string_field_24', 'string_field_25', 'string_field_26', 'string_field_27', 'string_field_28', 'string_field_29', 'string_field_30', 'string_field_31', 'string_field_32', 'string_field_33', 'string_field_34', 'string_field_35', 'string_field_36', 'string_field_37', 'string_field_38', 'string_field_39', 'string_field_40', 'string_field_41', 'string_field_42', 'string_field_43', 'string_field_44', 'string_field_45', 'string_field_46', 'string_field_47', 'string_field_48', 'string_field_49']


In [ ]:
# Check first row to understand all 50 column values
print(df.iloc[0].to_string())

string_field_0                                                H7323 
string_field_1                                            Local CCP 
string_field_2           WELLCARE NATIONAL HEALTH INSURANCE COMPANY 
string_field_3                                             Wellcare 
string_field_4                                  Centene Corporation 
string_field_5                                                   61%
string_field_6                                                   61%
string_field_7                                                   65%
string_field_8                            Not enough data available 
string_field_9                            Not enough data available 
string_field_10                                                  51%
string_field_11                 Plan not required to report measure 
string_field_12                 Plan not required to report measure 
string_field_13                 Plan not required to report measure 
string_field_14                   

In [ ]:
# Rename key columns and work with the data
df.rename(columns={
    'string_field_0': 'Contract_ID',
    'string_field_1': 'Organization_Type',
    'string_field_2': 'Contract_Name',
    'string_field_3': 'Marketing_Name',
    'string_field_4': 'Parent_Organization'
}, inplace=True)

# Get the header row from BigQuery to find measure names
query_header = """
SELECT *
FROM medicare-stars-2026.stars_data.measure_scores
WHERE string_field_0 = 'Contract ID'
LIMIT 1
"""
df_header = client.query(query_header).to_dataframe()
print(df_header.to_string())

Empty DataFrame
Columns: [string_field_0, string_field_1, string_field_2, string_field_3, string_field_4, string_field_5, string_field_6, string_field_7, string_field_8, string_field_9, string_field_10, string_field_11, string_field_12, string_field_13, string_field_14, string_field_15, string_field_16, string_field_17, string_field_18, string_field_19, string_field_20, string_field_21, string_field_22, string_field_23, string_field_24, string_field_25, string_field_26, string_field_27, string_field_28, string_field_29, string_field_30, string_field_31, string_field_32, string_field_33, string_field_34, string_field_35, string_field_36, string_field_37, string_field_38, string_field_39, string_field_40, string_field_41, string_field_42, string_field_43, string_field_44, string_field_45, string_field_46, string_field_47, string_field_48, string_field_49]
Index: []


In [ ]:
# Upload CSV file directly to Colab
from google.colab import files
uploaded = files.upload()

Saving 2026 Star Ratings Data Table - Measure Data (Oct 8 2025).csv to 2026 Star Ratings Data Table - Measure Data (Oct 8 2025).csv


In [ ]:
import pandas as pd
import io

# Load the uploaded file
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]), skiprows=2)

print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")
print(df.columns.tolist()[:10])
print(df.head(3))

Total rows: 770
Total columns: 50
['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'C01: Breast Cancer Screening', 'C02: Colorectal Cancer Screening', 'C03: Annual Flu Vaccine', 'C04: Improving or Maintaining Physical Health', 'C05: Improving or Maintaining Mental Health']
  Unnamed: 0                                Unnamed: 1          Unnamed: 2  \
0        NaN                                       NaN                 NaN   
1     E3014   Employer/Union Only Direct Contract PDP   PSERS HOP PROGRAM    
2     H0028                                 Local CCP       CHA HMO, INC.    

                                          Unnamed: 3  \
0                                                NaN   
1  Pennsylvania Public School Employees Retiremen...   
2                                            Humana    

                                          Unnamed: 4  \
0                                                NaN   
1  Commonwealth of PA Pub Schools Retirement System   

In [ ]:
# Clean and reshape the data
# Skip the first 2 metadata rows
df_clean = pd.read_csv(io.BytesIO(uploaded[filename]), skiprows=2)

# First column is the measure name/info
# Rows are contracts, columns are measures
print("Shape:", df_clean.shape)
print("\nFirst column values:")
print(df_clean.iloc[:, 0].head(20))

Shape: (770, 50)

First column values:
0        NaN
1     E3014 
2     H0028 
3     H0029 
4     H0034 
5     H0062 
6     H0074 
7     H0104 
8     H0107 
9     H0111 
10    H0154 
11    H0169 
12    H0174 
13    H0251 
14    H0270 
15    H0292 
16    H0294 
17    H0302 
18    H0321 
19    H0332 
Name: Unnamed: 0, dtype: object


In [ ]:
# Reload with correct header row
df_raw = pd.read_csv(io.BytesIO(uploaded[filename]), skiprows=2, header=0)

# Set contract ID as first column
df_raw.rename(columns={df_raw.columns[0]: 'Contract_ID'}, inplace=True)

# Drop row 0 (NaN row)
df_raw = df_raw.dropna(subset=['Contract_ID'])

# Melt from wide to long format
df_long = df_raw.melt(
    id_vars=['Contract_ID'],
    var_name='Measure_Name',
    value_name='Score'
)

# Clean score column
df_long['Score_Numeric'] = pd.to_numeric(
    df_long['Score'].str.replace('%', '').str.strip(),
    errors='coerce'
)

# Remove invalid rows
df_long = df_long.dropna(subset=['Score_Numeric'])

print(f"Total records: {len(df_long)}")
print(df_long.head(10))

Total records: 21273
     Contract_ID                  Measure_Name Score  Score_Numeric
3077      H0028   C01: Breast Cancer Screening   76%           76.0
3079      H0034   C01: Breast Cancer Screening   61%           61.0
3081      H0074   C01: Breast Cancer Screening   75%           75.0
3082      H0104   C01: Breast Cancer Screening   78%           78.0
3083      H0107   C01: Breast Cancer Screening   80%           80.0
3084      H0111   C01: Breast Cancer Screening   71%           71.0
3085      H0154   C01: Breast Cancer Screening   77%           77.0
3086      H0169   C01: Breast Cancer Screening   70%           70.0
3087      H0174   C01: Breast Cancer Screening   69%           69.0
3088      H0251   C01: Breast Cancer Screening   65%           65.0


In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

# Analysis 1 — Performance Categories
df_long['Performance_Category'] = df_long['Score_Numeric'].apply(
    lambda x: 'Below Threshold' if x < 60 else ('At Risk' if x <= 75 else 'Performing Well')
)

# Analysis 2 — Worst performing measures
measure_avg = df_long.groupby('Measure_Name')['Score_Numeric'].mean().reset_index()
measure_avg.columns = ['Measure_Name', 'Avg_Score']
measure_avg = measure_avg.sort_values('Avg_Score')

# Chart 1 — Bottom 10 measures
plt.figure(figsize=(12, 6))
bottom10 = measure_avg.head(10)
plt.barh(bottom10['Measure_Name'], bottom10['Avg_Score'], color='red')
plt.xlabel('Average Score (%)')
plt.title('Bottom 10 HEDIS Measures by Average Score — 2026 CMS Star Ratings')
plt.tight_layout()
plt.savefig('bottom_10_measures.png', dpi=150)
plt.show()
print("Chart 1 saved!")

# Chart 2 — Performance category distribution
plt.figure(figsize=(8, 5))
category_counts = df_long['Performance_Category'].value_counts()
colors = ['red', 'orange', 'green']
plt.pie(category_counts, labels=category_counts.index, autopct='%1.1f%%', colors=colors)
plt.title('HEDIS Measure Performance Distribution — 2026')
plt.savefig('performance_distribution.png', dpi=150)
plt.show()
print("Chart 2 saved!")

print(f"\nBelow Threshold (<60%): {len(df_long[df_long['Performance_Category']=='Below Threshold'])}")
print(f"At Risk (60-75%): {len(df_long[df_long['Performance_Category']=='At Risk'])}")
print(f"Performing Well (>75%): {len(df_long[df_long['Performance_Category']=='Performing Well'])}")

Chart 1 saved!
Chart 2 saved!

Below Threshold (<60%): 5166
At Risk (60-75%): 2679
Performing Well (>75%): 13428


In [ ]:
# Save clean data as CSV for Tableau and Power BI
df_long.to_csv('hedis_stars_clean.csv', index=False)

# Save measure summary
measure_summary = df_long.groupby('Measure_Name').agg(
    Total_Plans=('Contract_ID', 'count'),
    Avg_Score=('Score_Numeric', 'mean'),
    Min_Score=('Score_Numeric', 'min'),
    Max_Score=('Score_Numeric', 'max'),
    Below_Threshold=('Performance_Category', lambda x: (x=='Below Threshold').sum()),
    At_Risk=('Performance_Category', lambda x: (x=='At Risk').sum()),
    Performing_Well=('Performance_Category', lambda x: (x=='Performing Well').sum())
).reset_index()

measure_summary.to_csv('measure_summary.csv', index=False)

# Download both files
from google.colab import files
files.download('hedis_stars_clean.csv')
files.download('measure_summary.csv')

print("Files downloaded successfully!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Files downloaded successfully!
